# Lab 15: Mini-StyleGAN on MNIST

**Goal:** Build a simplified StyleGAN to see the key ideas in action:

1. Mapping network (z → w)
2. AdaIN (style injection at every layer)
3. Noise injection (stochastic details)
4. Train it on MNIST
5. Style mixing (swap styles between layers)
6. Truncation trick (quality vs diversity)

This is a *teaching* StyleGAN — small enough to train on CPU/basic GPU,
big enough to demonstrate every concept from the lesson.

**Sizes vs the paper:** StyleGAN uses z, w ∈ R^512 and an 8-layer mapping network at 1024×1024. Here z = 64 (as in the Unit 2 labs), w = 128, 4 mapping layers, 28×28 digits.

**Expected compute** (approximate): 30 epochs ≈ 20–30 min on Apple-silicon (`mps`) or a Colab T4; on CPU the notebook trains on a 10k-image subset (≈ 60–90 min). The experiments afterwards take seconds.

**How to use this lab:** each experiment has a **Predict** prompt — answer it before you run the cell.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available()  # Apple-silicon GPU
                      else "cpu")
print(f"Using: {device}")

## 1. The Data

Same MNIST setup as always. 28x28 grayscale, normalized to [-1, 1].

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

mnist = datasets.MNIST("data", train=True, download=True, transform=transform)
# CPU-only machines train on a 10k-image subset (None = all 60k images).
TRAIN_SUBSET = 10_000 if device.type == "cpu" else None
train_set = torch.utils.data.Subset(mnist, range(TRAIN_SUBSET)) if TRAIN_SUBSET else mnist
loader = DataLoader(train_set, batch_size=64, shuffle=True, drop_last=True)

print(f"Dataset: {len(mnist)} images")
print(f"Image shape: {mnist[0][0].shape}")

## 2. Building Block: AdaIN

The core of StyleGAN. Two steps:
1. **Normalize** — strip existing style (instance norm)
2. **Re-style** — apply new style from w (learned scale + shift)

The scale is written as `1 + γ`: at initialization γ ≈ 0, so AdaIN starts close to plain instance norm instead of multiplying every feature by ≈ 0. (StyleGAN initializes its style scale to 1 for the same reason.)

In [ ]:
class AdaIN(nn.Module):
    """Adaptive Instance Normalization.
    Takes a feature map and a style vector w.
    Normalizes the feature map, then re-scales/shifts using w."""

    def __init__(self, w_dim, channels):
        super().__init__()
        self.style = nn.Linear(w_dim, channels * 2)  # gamma + beta

    def forward(self, x, w):
        # x: (batch, channels, H, W)
        # w: (batch, w_dim)
        style = self.style(w)                      # (batch, channels*2)
        gamma, beta = style.chunk(2, dim=1)        # each (batch, channels)
        gamma = 1 + gamma.unsqueeze(2).unsqueeze(3)  # (batch, channels, 1, 1); scale starts near 1
        beta  = beta.unsqueeze(2).unsqueeze(3)

        # Instance normalize
        mean = x.mean(dim=[2, 3], keepdim=True)
        std  = x.std(dim=[2, 3], keepdim=True) + 1e-8
        x_norm = (x - mean) / std

        # Apply style
        return gamma * x_norm + beta

# Quick test
adain = AdaIN(w_dim=128, channels=64)
test_x = torch.randn(2, 64, 7, 7)
test_w = torch.randn(2, 128)
out = adain(test_x, test_w)
print(f"AdaIN: input {test_x.shape} + style {test_w.shape} -> output {out.shape}")

## 3. Mapping Network

8 FC layers that transform z (random noise) into w (disentangled style).

We use 4 layers here (smaller model for MNIST).

In [ ]:
class MappingNetwork(nn.Module):
    def __init__(self, z_dim, w_dim, num_layers=4):
        super().__init__()
        layers = []
        for i in range(num_layers):
            in_dim = z_dim if i == 0 else w_dim
            layers.append(nn.Linear(in_dim, w_dim))
            layers.append(nn.LeakyReLU(0.2))
        self.net = nn.Sequential(*layers)

    def forward(self, z):
        return self.net(z)

# Test
z_dim = 64
w_dim = 128
mapping = MappingNetwork(z_dim, w_dim)
z = torch.randn(4, z_dim)
w = mapping(z)
print(f"Mapping: z {z.shape} -> w {w.shape}")

## 4. Style Block

One layer of the StyleGAN synthesis network:
- Conv → Noise injection → AdaIN → LeakyReLU

In [ ]:
class StyleBlock(nn.Module):
    """One layer: conv -> noise -> AdaIN -> activation."""

    def __init__(self, in_ch, out_ch, w_dim, upsample=False):
        super().__init__()
        self.upsample = upsample

        if upsample:
            self.conv = nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1)
        else:
            self.conv = nn.Conv2d(in_ch, out_ch, 3, padding=1)

        self.adain = AdaIN(w_dim, out_ch)
        self.noise_scale = nn.Parameter(torch.zeros(1, out_ch, 1, 1))
        self.act = nn.LeakyReLU(0.2)

    def forward(self, x, w):
        x = self.conv(x)

        # Noise injection
        noise = torch.randn(x.size(0), 1, x.size(2), x.size(3), device=x.device)
        x = x + self.noise_scale * noise

        # AdaIN
        x = self.adain(x, w)
        x = self.act(x)
        return x

# Test
block = StyleBlock(64, 32, w_dim=128, upsample=True)
test_x = torch.randn(2, 64, 7, 7)
test_w = torch.randn(2, 128)
out = block(test_x, test_w)
print(f"StyleBlock (upsample): {test_x.shape} -> {out.shape}")

## 5. Mini-StyleGAN Generator

Key differences from DCGAN:
1. Starts from a **learned constant** (not from z)
2. z → mapping network → w
3. w injected at **every layer** via AdaIN
4. Noise injected at every layer

Architecture: constant 4×4 → 4×4 → 8×8 → 16×16 → 32×32, then crop to 28×28

In [ ]:
class StyleGenerator(nn.Module):
    def __init__(self, z_dim=64, w_dim=128):
        super().__init__()
        self.w_dim = w_dim
        self.num_layers = 4  # number of style injection points

        # Mapping network: z -> w
        self.mapping = MappingNetwork(z_dim, w_dim)

        # Learned constant (replaces z as the starting point)
        # Shape: (1, 256, 4, 4) — will be expanded per batch
        self.constant = nn.Parameter(torch.randn(1, 256, 4, 4))

        # Style blocks: 4x4 -> 4x4 -> 8x8 -> 16x16 -> 32x32 (cropped to 28x28)
        self.style1 = StyleBlock(256, 256, w_dim)           # 4x4 -> 4x4  (coarse)
        self.style2 = StyleBlock(256, 128, w_dim, upsample=True)  # 4x4 -> 8x8
        self.style3 = StyleBlock(128, 64, w_dim, upsample=True)   # 8x8 -> 16x16
        self.style4 = StyleBlock(64, 32, w_dim, upsample=True)    # 16x16 -> 32x32

        # To RGB (final 1x1 conv, no style)
        self.to_rgb = nn.Sequential(
            nn.Conv2d(32, 1, 1),
            nn.Tanh(),
        )

    def forward(self, z, return_w=False):
        """Normal forward: one z, one w for all layers."""
        w = self.mapping(z)
        img = self.synthesize([w] * self.num_layers)
        if return_w:
            return img, w
        return img

    def synthesize(self, w_list):
        """Generate image from a list of w vectors (one per layer).
        This enables style mixing: different w at different layers."""
        batch_size = w_list[0].size(0)
        x = self.constant.expand(batch_size, -1, -1, -1)  # (batch, 256, 4, 4)

        x = self.style1(x, w_list[0])   # 4x4,  coarse
        x = self.style2(x, w_list[1])   # 8x8,  coarse
        x = self.style3(x, w_list[2])   # 16x16, medium
        x = self.style4(x, w_list[3])   # 32x32, fine

        # Crop to 28x28 (since 32x32 from upsampling)
        x = x[:, :, 2:30, 2:30]

        return self.to_rgb(x)

# Test
gen = StyleGenerator(z_dim, w_dim).to(device)
z = torch.randn(4, z_dim, device=device)
img = gen(z)
print(f"Generator output: {img.shape}")
print(f"Parameters: {sum(p.numel() for p in gen.parameters()):,}")

## 6. Discriminator

Standard convolutional discriminator (nothing special here — StyleGAN's innovation is in the Generator).

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(1, 64, 4, 2, 1),       # 28 -> 14
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, 2, 1),      # 14 -> 7
            nn.InstanceNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, 4, 2, 1),     # 7 -> 3 (with padding)
            nn.InstanceNorm2d(256),
            nn.LeakyReLU(0.2),
            nn.Flatten(),
            nn.Linear(256 * 3 * 3, 1),
        )

    def forward(self, x):
        return self.model(x)

disc = Discriminator().to(device)
test = torch.randn(4, 1, 28, 28, device=device)
print(f"Discriminator output: {disc(test).shape}")
print(f"Parameters: {sum(p.numel() for p in disc.parameters()):,}")

## 7. Training with WGAN-GP Loss

StyleGAN uses a non-saturating GAN loss with R1 regularization.
We'll use WGAN-GP (which we already know) for stability.

In [ ]:
def gradient_penalty(disc, real, fake, device):
    """WGAN-GP gradient penalty."""
    bs = real.size(0)
    alpha = torch.rand(bs, 1, 1, 1, device=device)
    interpolated = (alpha * real + (1 - alpha) * fake).requires_grad_(True)
    d_out = disc(interpolated)
    grads = torch.autograd.grad(
        outputs=d_out,
        inputs=interpolated,
        grad_outputs=torch.ones_like(d_out),
        create_graph=True,
    )[0]
    grads = grads.view(bs, -1)
    return ((grads.norm(2, dim=1) - 1) ** 2).mean()

print("Gradient penalty function ready.")

In [ ]:
# Fresh models
gen  = StyleGenerator(z_dim, w_dim).to(device)
disc = Discriminator().to(device)

# Adam for both (StyleGAN uses Adam, not RMSprop)
opt_G = torch.optim.Adam(gen.parameters(), lr=1e-4, betas=(0.0, 0.99))
opt_D = torch.optim.Adam(disc.parameters(), lr=1e-4, betas=(0.0, 0.99))

gp_weight = 10
n_critic = 5
epochs = 30

d_losses, g_losses = [], []

print(f"Training Mini-StyleGAN for {epochs} epochs...")
print(f"Critic steps per G step: {n_critic}")
print()

In [ ]:
for epoch in range(epochs):
    d_loss_sum, g_loss_sum = 0, 0
    d_steps, g_steps = 0, 0

    for batch_idx, (real_imgs, _) in enumerate(loader):
        real_imgs = real_imgs.to(device)
        bs = real_imgs.size(0)

        # === Critic step ===
        z = torch.randn(bs, z_dim, device=device)
        fake = gen(z).detach()

        d_real = disc(real_imgs).mean()
        d_fake = disc(fake).mean()
        gp = gradient_penalty(disc, real_imgs, fake, device)

        loss_D = d_fake - d_real + gp_weight * gp

        opt_D.zero_grad()
        loss_D.backward()
        opt_D.step()

        d_loss_sum += loss_D.item()
        d_steps += 1

        # === Generator step (every n_critic batches) ===
        if (batch_idx + 1) % n_critic == 0:
            z = torch.randn(bs, z_dim, device=device)
            fake = gen(z)
            loss_G = -disc(fake).mean()

            opt_G.zero_grad()
            loss_G.backward()
            opt_G.step()

            g_loss_sum += loss_G.item()
            g_steps += 1

    avg_d = d_loss_sum / max(d_steps, 1)
    avg_g = g_loss_sum / max(g_steps, 1)
    d_losses.append(avg_d)
    g_losses.append(avg_g)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{epochs} | D: {avg_d:+.4f} | G: {avg_g:+.4f}")

print("\nTraining complete!")

In [ ]:
# Loss curves
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(d_losses, label="Critic", alpha=0.8)
ax.plot(g_losses, label="Generator", alpha=0.8)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Mini-StyleGAN Training")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Show generated samples
gen.eval()
with torch.no_grad():
    z = torch.randn(16, z_dim, device=device)
    samples = gen(z).cpu()

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    img = (samples[i].squeeze() + 1) / 2
    ax.imshow(img, cmap="gray")
    ax.axis("off")
plt.suptitle("Mini-StyleGAN Generated Digits", fontsize=14)
plt.tight_layout()
plt.show()

## 8. Experiment 1: Style Mixing

The whole point of StyleGAN's architecture: use **different** w vectors at different layers.

- **Coarse layers** (style1, style2): overall digit shape/identity
- **Fine layers** (style3, style4): stroke texture, thickness, details

**Predict:** in the grid, should rows (same coarse w) or columns (same fine w) share the digit's identity?

In [ ]:
gen.eval()

n_sources = 4  # source A images (rows)
n_targets = 4  # source B images (cols)

with torch.no_grad():
    # Two sets of z vectors
    z_A = torch.randn(n_sources, z_dim, device=device)
    z_B = torch.randn(n_targets, z_dim, device=device)

    # Map to w
    w_A = gen.mapping(z_A)  # (n_sources, w_dim)
    w_B = gen.mapping(z_B)  # (n_targets, w_dim)

    # Generate pure A and pure B
    imgs_A = gen.synthesize([w_A] * gen.num_layers).cpu()
    imgs_B = gen.synthesize([w_B] * gen.num_layers).cpu()

    # Style mixing: coarse from A, fine from B
    mixed_coarse = []  # A's shape + B's texture
    for i in range(n_sources):
        row = []
        for j in range(n_targets):
            wa = w_A[i:i+1]  # (1, w_dim)
            wb = w_B[j:j+1]
            # Coarse (layers 0,1) from A, Fine (layers 2,3) from B
            w_list = [wa, wa, wb, wb]
            mixed = gen.synthesize(w_list).cpu()
            row.append(mixed)
        mixed_coarse.append(row)

# Plot style mixing grid
fig, axes = plt.subplots(n_sources + 1, n_targets + 1, figsize=(10, 10))

# Top-left: empty
axes[0, 0].axis("off")
axes[0, 0].text(0.5, 0.5, "Coarse\n(A)\n↓", ha='center', va='center',
                fontsize=10, fontweight='bold')

# Top row: source B (fine style)
for j in range(n_targets):
    axes[0, j+1].imshow((imgs_B[j].squeeze() + 1)/2, cmap="gray")
    axes[0, j+1].axis("off")
    if j == 0:
        axes[0, j+1].set_title("Fine (B) →", fontsize=10, fontweight='bold')

# Left column: source A (coarse style)
for i in range(n_sources):
    axes[i+1, 0].imshow((imgs_A[i].squeeze() + 1)/2, cmap="gray")
    axes[i+1, 0].axis("off")

# Grid: mixed images
for i in range(n_sources):
    for j in range(n_targets):
        axes[i+1, j+1].imshow((mixed_coarse[i][j].squeeze() + 1)/2, cmap="gray")
        axes[i+1, j+1].axis("off")

plt.suptitle("Style Mixing: Coarse from A (rows), Fine from B (columns)", fontsize=13)
plt.tight_layout()
plt.show()

print("Each grid cell = A's coarse structure + B's fine texture")
print("If the styles are disentangled, rows share shape, columns share texture.")

## 9. Experiment 2: Truncation Trick

Move w toward the mean for higher quality, less diversity.

`w_truncated = w_mean + psi * (w - w_mean)`

- psi=1.0: full diversity (no truncation)
- psi=0.5: less diversity, potentially sharper
- psi=0.0: always the "average" digit

You met truncation in **z** in L11 (re-sample |z_i| > t). StyleGAN truncates in **w** instead — same quality-vs-variety trade-off.

**Predict:** at psi = 0, how many different images will the 8 columns show?

In [ ]:
# Step 1: Compute w_mean over many samples
with torch.no_grad():
    z_many = torch.randn(5000, z_dim, device=device)
    w_many = gen.mapping(z_many)
    w_mean = w_many.mean(dim=0, keepdim=True)  # (1, w_dim)

print(f"w_mean shape: {w_mean.shape}")
print(f"w_mean range: [{w_mean.min():.2f}, {w_mean.max():.2f}]")

In [ ]:
# Step 2: Generate with different psi values
psi_values = [0.0, 0.3, 0.5, 0.7, 1.0, 1.5]
n_samples = 8

# Use the SAME z for all psi values (so we can compare)
z_fixed = torch.randn(n_samples, z_dim, device=device)

fig, axes = plt.subplots(len(psi_values), n_samples, figsize=(12, 9))

with torch.no_grad():
    w_fixed = gen.mapping(z_fixed)  # (n_samples, w_dim)

    for row, psi in enumerate(psi_values):
        # Truncation
        w_trunc = w_mean + psi * (w_fixed - w_mean)
        imgs = gen.synthesize([w_trunc] * gen.num_layers).cpu()

        for col in range(n_samples):
            axes[row, col].imshow((imgs[col].squeeze() + 1)/2, cmap="gray")
            axes[row, col].axis("off")

        axes[row, 0].set_ylabel(f"psi={psi}", fontsize=11, rotation=0, labelpad=45)

plt.suptitle("Truncation Trick: psi=0 (all same) → psi=1.5 (max diversity)", fontsize=13)
plt.tight_layout()
plt.show()

print("psi=0.0: all images converge to the 'average' digit")
print("psi=1.0: normal generation (no truncation)")
print("psi=1.5: extra diversity (may get weird artifacts)")

## 10. Experiment 3: Noise Controls Stochastic Details

The **same w** with **different noise** should produce the same digit
but with slightly different fine details.

**Predict:** will the digit change across a row, or only its strokes?

In [ ]:
# Generate multiple images from the same w (different noise each time)
n_variations = 8
n_styles = 4

fig, axes = plt.subplots(n_styles, n_variations, figsize=(12, 6))

with torch.no_grad():
    for row in range(n_styles):
        # One z → one w
        z_single = torch.randn(1, z_dim, device=device)
        w_single = gen.mapping(z_single)  # (1, w_dim)

        for col in range(n_variations):
            # Same w, but forward pass generates NEW random noise each time
            img = gen.synthesize([w_single] * gen.num_layers).cpu()
            axes[row, col].imshow((img.squeeze() + 1)/2, cmap="gray")
            axes[row, col].axis("off")

        axes[row, 0].set_ylabel(f"Style {row+1}", fontsize=11, rotation=0, labelpad=45)

plt.suptitle("Same Style (w), Different Noise → Same digit, different details", fontsize=13)
plt.tight_layout()
plt.show()

print("Each row = same w (same digit identity)")
print("Each column = different random noise (different fine details)")
print("The differences should be subtle — stroke thickness, edge roughness, etc.")

## 11. Experiment 4: W-Space Interpolation

Interpolate between two digits in W-space.
Should be smoother than Z-space interpolation (from L11).

**Predict:** which rows will show more sudden jumps — Z or W?

In [ ]:
n_steps = 10
n_pairs = 3

fig, axes = plt.subplots(n_pairs * 2, n_steps, figsize=(14, 9))

with torch.no_grad():
    for pair in range(n_pairs):
        z1 = torch.randn(1, z_dim, device=device)
        z2 = torch.randn(1, z_dim, device=device)
        w1 = gen.mapping(z1)
        w2 = gen.mapping(z2)

        for step in range(n_steps):
            t = step / (n_steps - 1)

            # Z-space interpolation
            z_interp = (1 - t) * z1 + t * z2
            img_z = gen(z_interp).cpu()

            # W-space interpolation
            w_interp = (1 - t) * w1 + t * w2
            img_w = gen.synthesize([w_interp] * gen.num_layers).cpu()

            row_z = pair * 2
            row_w = pair * 2 + 1

            axes[row_z, step].imshow((img_z.squeeze() + 1)/2, cmap="gray")
            axes[row_z, step].axis("off")
            axes[row_w, step].imshow((img_w.squeeze() + 1)/2, cmap="gray")
            axes[row_w, step].axis("off")

        axes[pair * 2, 0].set_ylabel("Z-space", fontsize=10, rotation=0, labelpad=50)
        axes[pair * 2 + 1, 0].set_ylabel("W-space", fontsize=10, rotation=0, labelpad=50)

plt.suptitle("Z-space vs W-space Interpolation", fontsize=14)
plt.tight_layout()
plt.show()

print("W-space interpolation should produce smoother transitions.")
print("Z-space may have 'jumps' or low-quality intermediate images.")

## 12. Experiment 5: What Does the Learned Constant Look Like?

In DCGAN, the starting point comes from z. In StyleGAN, it's a **learned constant**.
Let's visualize it.

In [ ]:
# The constant is a (1, 256, 4, 4) tensor
constant = gen.constant.data.cpu().squeeze()  # (256, 4, 4)

# Show first 16 channels
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(constant[i], cmap="viridis")
    ax.axis("off")
    ax.set_title(f"ch {i}", fontsize=9)

plt.suptitle("Learned Constant: First 16 channels (4x4 each)", fontsize=13)
plt.tight_layout()
plt.show()

print(f"Constant shape: {gen.constant.shape}")
print(f"Value range: [{constant.min():.2f}, {constant.max():.2f}]")
print("\nThis is the SAME starting point for every generated image.")
print("All diversity comes from w (via AdaIN) and noise injections.")

## 13. Experiment 6: Noise Scale Analysis

How much does each layer's noise affect the output?

In [ ]:
# Print learned noise scales
print("Learned noise scales per layer:")
print("(Larger = noise has more effect at this layer)\n")

for name, layer in [("Style 1 (4x4, coarse)", gen.style1),
                     ("Style 2 (8x8, coarse)", gen.style2),
                     ("Style 3 (16x16, medium)", gen.style3),
                     ("Style 4 (32x32, fine)", gen.style4)]:
    scale = layer.noise_scale.data.abs().mean().item()
    bar = "|" * int(scale * 500)
    print(f"  {name:30s}  {scale:.5f}  {bar}")

print("\nFine layers typically learn larger noise scales,")
print("because noise controls fine-grained stochastic details.")

## Key Takeaways

1. **Mapping network** turns z into w — disentangles features
2. **AdaIN** injects style at every layer — normalize, then re-scale/shift
3. **Learned constant** replaces z as starting point — all variation from w + noise
4. **Style mixing** — different w at different layers mixes structures
5. **Truncation trick** — move w toward w_mean for higher quality
6. **Noise injection** — controls random details (hair, texture), not identity
7. **W-space interpolation** — smoother than Z-space

This is a mini version. Real StyleGAN trains on 1024x1024 faces for days on 8 GPUs.
But every concept here — mapping network, AdaIN, noise, style mixing — is identical.

**Unit 3 complete! Next: Unit 4 — Image-to-image translation (Pix2Pix; CycleGAN in Unit 5)**